# 06_採購_供應商報價

同條件比較與最低值

任務：計算每筆總成本＝單價 × 訂購數量＋運費。；比較全部報價的最低總成本。；加入交期不超過 3 天的條件，再找最低成本。

規則：品項規格與數量相同，價格視為含稅；交期是教學約定，成本相同時保留所有並列者。

先人工核對再執行。資料皆為教學模擬值。

In [ ]:
%pip -q install openpyxl numpy matplotlib

In [ ]:
from pathlib import Path
import csv
from openpyxl import load_workbook

# 本機：把 Notebook 與 Excel 放在同一資料夾。Colab：先上傳本例 Excel。
SOURCE = Path('06_採購_供應商報價.xlsx')
if not SOURCE.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'請將 {SOURCE.name} 放到目前工作目錄：{Path.cwd()}')
    files.upload()
if not SOURCE.exists():
    raise FileNotFoundError(f'尚未找到 {SOURCE.name}，請確認上傳檔名')
OUT = Path('output') / '06_採購_供應商報價'
OUT.mkdir(parents=True, exist_ok=True)

def read_rows(sheet):
    data = sheet.iter_rows(values_only=True)
    headers = next(data)
    return [dict(zip(headers, row)) for row in data if any(v is not None for v in row)]

def save_csv(name, records):
    if not records:
        print('沒有符合條件的資料，未輸出 CSV')
        return
    with (OUT / name).open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

# 來源檔只有原始值，沒有需要重算的 Excel 公式。
wb = load_workbook(SOURCE, read_only=True, data_only=True)
print('工作表：', wb.sheetnames)
rows = read_rows(wb['資料'])
print('第一筆：', rows[0])

In [ ]:
result = [dict(r, 總成本_元=r['單價_元'] * r['訂購數量'] + r['運費_元']) for r in rows]
lowest = min(r['總成本_元'] for r in result)
eligible = [r for r in result if r['交期_天'] <= 3]
fastest_cost = min(r['總成本_元'] for r in eligible)
print('全體最低：', [r for r in result if r['總成本_元'] == lowest])
print('交期內最低：', [r for r in eligible if r['總成本_元'] == fastest_cost])
assert lowest == 2500 and fastest_cost == 2600
wb.close()

## 驗收答案

全體最低丙供應商 2,500 元；交期 ≤ 3 天時，甲供應商 2,600 元。